<a href="https://colab.research.google.com/github/Simran-12005/TableTalk_GSoc_2026/blob/main/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Audio_Speech_Actors_01-24.zip to Audio_Speech_Actors_01-24.zip


In [15]:
import zipfile

zip_path = "/content/dataset/Audio_Speech_Actors_01-24.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [16]:
import os

for root, dirs, files in os.walk("/content/dataset"):
    for file in files[:10]:
        print(os.path.join(root, file))

/content/dataset/Audio_Speech_Actors_01-24.zip
/content/dataset/Actor_11/03-01-07-01-01-01-11.wav
/content/dataset/Actor_11/03-01-02-01-01-01-11.wav
/content/dataset/Actor_11/03-01-01-01-01-01-11.wav
/content/dataset/Actor_11/03-01-07-02-01-01-11.wav
/content/dataset/Actor_11/03-01-06-02-01-02-11.wav
/content/dataset/Actor_11/03-01-07-01-02-01-11.wav
/content/dataset/Actor_11/03-01-08-02-01-02-11.wav
/content/dataset/Actor_11/03-01-05-01-01-01-11.wav
/content/dataset/Actor_11/03-01-08-01-02-01-11.wav
/content/dataset/Actor_11/03-01-03-02-01-02-11.wav
/content/dataset/Actor_06/03-01-03-01-02-02-06.wav
/content/dataset/Actor_06/03-01-01-01-02-01-06.wav
/content/dataset/Actor_06/03-01-05-01-01-01-06.wav
/content/dataset/Actor_06/03-01-03-02-01-01-06.wav
/content/dataset/Actor_06/03-01-06-01-02-02-06.wav
/content/dataset/Actor_06/03-01-06-02-01-01-06.wav
/content/dataset/Actor_06/03-01-07-02-02-01-06.wav
/content/dataset/Actor_06/03-01-02-01-01-02-06.wav
/content/dataset/Actor_06/03-01-02-

In [21]:
import os

wav_count = 0

for root, dirs, files in os.walk("/content/dataset"):
    for file in files:
        if file.endswith(".wav"):
            wav_count += 1

print("WAV files found:", wav_count)

WAV files found: 1440


In [22]:
for root, dirs, files in os.walk("/content/dataset"):
    for file in files:
        if file.endswith(".wav"):
            print("Example file:", os.path.join(root, file))
            break
    break

In [23]:
test_file = "/content/dataset/Actor_01/03-01-01-01-01-01-01.wav"

features = extract_features(test_file)
print("Feature length:", len(features))

Feature length: 17


In [32]:
data = []
count = 0
MAX_FILES = 100   # limit

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith(".wav"):
            file_path = os.path.join(root, file)

            try:
                features = extract_features(file_path)
                label = get_label(file)

                data.append([file, label] + features)

                count += 1

                if count >= MAX_FILES:
                    break

            except Exception as e:
                print("Error:", e)

    if count >= MAX_FILES:
        break

print("Processed files:", count)

Processed files: 100


In [33]:
df = pd.DataFrame(data, columns=columns)
print(df.shape)
df.head()

(100, 19)


,filename,label,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,pitch,energy,spectral_centroid,duration
0,03-01-07-01-01-01-11.wav,7,-320.139435,60.783325,14.142775,17.426476,2.209573,2.983834,2.639669,-2.570144,-7.192221,-4.735665,5.276324,-3.230752,-3.730215,1630.684326,0.075655,2733.516758,3.0
1,03-01-02-01-01-01-11.wav,2,-310.752472,66.281715,24.678993,23.338121,16.507885,12.499136,8.958508,4.132096,-0.461540,4.894765,6.281127,1.723106,-2.385299,1347.941772,0.079430,2678.800432,3.0
2,03-01-01-01-01-01-11.wav,1,-374.563385,64.256592,24.880774,20.290344,12.468756,5.764773,7.012583,1.688628,0.067165,0.669672,4.003408,3.018394,-0.266998,1228.738525,0.065670,2664.110055,3.0
3,03-01-07-02-01-01-11.wav,7,-295.874603,57.526787,8.880239,16.831015,0.874271,1.424733,1.067272,-3.586244,-9.498210,-3.571720,6.294766,-3.075656,-2.354086,1654.138794,0.075457,2825.174684,3.0
4,03-01-06-02-01-02-11.wav,6,-336.549225,37.330334,-4.041380,5.401594,1.104917,-2.099465,-1.542329,-2.479206,-8.260303,-2.578178,-0.046116,-3.003858,-5.870682,1593.601562,0.072382,2751.308821,3.0


In [29]:
import librosa
import numpy as np
import pandas as pd
import os

DATASET_PATH = "/content/dataset"

data = []

# Extract emotion label from filename (RAVDESS)
def get_label(filename):
    try:
        return int(filename.split("-")[2])
    except:
        return -1  # fallback if format differs

def extract_features(file_path):
    # Load audio (fixed duration)
    y, sr = librosa.load(file_path, duration=3)

    # Normalize
    y = librosa.util.normalize(y)

    # MFCC (13)
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)

    # Pitch
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0

    # Energy
    energy = np.mean(librosa.feature.rms(y=y))

    # Spectral centroid
    spec_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))

    # Duration
    duration = librosa.get_duration(y=y, sr=sr)

    return list(mfcc) + [pitch, energy, spec_centroid, duration]


# Process all files
for file in os.listdir(DATASET_PATH):
    if file.endswith(".wav"):
        file_path = os.path.join(DATASET_PATH, file)

        features = extract_features(file_path)
        label = get_label(file)

        data.append([file, label] + features)

# Columns
columns = ["filename", "label"] + [f"mfcc_{i}" for i in range(13)] + [
    "pitch", "energy", "spectral_centroid", "duration"
]

# DataFrame
df = pd.DataFrame(data, columns=columns)

# Save CSV
df.to_csv("features.csv", index=False)

print("Task 1 Completed Successfully!")
df.head()

Task 1 Completed Successfully!


,filename,label,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,pitch,energy,spectral_centroid,duration
